In [2]:
%reload_ext autoreload
%autoreload 2

In [3]:
import numpy as np
from topoflow.flow.paths import GaussianConditionalProbabilityPath, DiscreteConditionalProbabilityPath
from topoflow.flow.schedulers import LinearAlpha, LinearBeta
from topoflow.utils.visualization import visualize_data_rdkit, visualize_data, visualize_trajectory, visualize_data_matplotlib
from topoflow.flow.old_flow_models_discrete import FlowModel as FlowModelDiscrete
from topoflow.flow.flow_models import FlowModel
from topoflow.data.dataloader_adj import DataLoaderSampleable
from topoflow.data.datasets import CrossDocked
from topoflow.flow.schedulers import LinearAlpha, LinearBeta, CosineAlpha, CosineBeta
import torch
import lightning as pl
from torch_geometric.data import Data

/home/marco/miniforge3/envs/topoflow/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
batch_size = 32
data_dir = "../data/CrossDocked"
train_dataset = CrossDocked(
    root=data_dir,
    split="train",
    transform=None,
)
val_dataset = CrossDocked(root=data_dir, split="val", transform=None)

alpha = CosineAlpha()
beta = CosineBeta()

p_data = DataLoaderSampleable(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
)


distribution_hist = np.load("../data/CrossDocked/processed/size_distribution_full.npy", allow_pickle=True)
path = GaussianConditionalProbabilityPath(p_data=p_data, alpha=alpha, beta=beta, distribution_hist=distribution_hist, device="cuda" if torch.cuda.is_available() else "cpu")
discrete_path = DiscreteConditionalProbabilityPath(p_data=p_data, alpha=alpha, n_classes=11)
checkpoint_path = "../checkpoints/runs_cluster/to_evaluate/03_11_1199e_ast_gat_scaled_vel.ckpt"

#Load the model from the checkpoint
# model = FlowModelDiscrete.load_from_checkpoint(checkpoint_path, path=path, discrete_path=discrete_path, norm_path="../data/CrossDocked/processed/normalization_params_full.npz")
model = FlowModel.load_from_checkpoint(checkpoint_path, path=path, norm_path="../data/CrossDocked/processed/normalization_params.npz")

Loaded dataset from ../data/CrossDocked/processed/train_crossdocked_pyg.pt.
Loaded SMILES file from ../data/CrossDocked/processed/train_crossdocked_pyg_smiles.txt
Dataset loaded successfully with 512 samples.
Loaded dataset from ../data/CrossDocked/processed/val_crossdocked_pyg.pt.
Loaded SMILES file from ../data/CrossDocked/processed/val_crossdocked_pyg_smiles.txt
Dataset loaded successfully with 512 samples.


/home/marco/Documents/phd/topoFlow/topoflow/data/dataloader_adj.py:23: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  super().__init__(
/home/marco/miniforge3/envs/topoflow/lib/python3.10/site-packages/lightning/fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case wher

In [7]:
# Select a specific pocket from the dataset
pocket_sample = val_dataset[6]

camera = [
    -0.288512796,   -0.942787290,   -0.167020068,\
    -0.749116302,    0.113638930,    0.652612567,\
    -0.596299529,    0.313405633,   -0.739049375,\
    -0.000015971,   -0.000005450,  -57.064853668,\
    -0.109379508,   -1.517336011,   -1.236875176,\
    35.980468750,   78.151092529,  -20.000000000   ]

In [9]:
def create_interpolation_trajectory(data, path, n_steps=100, unnorm_fn=None):
    device = path.device
    trajectory = []
    # z: [n_atoms, n_coords + n_classes]
    z_coords = data.x_lig.to(device)
    z_atoms = data.h_lig.to(device)
    z = torch.cat([z_coords, z_atoms], dim=-1)
    
    # Sample initial noise w
    w = path.p_simple.sample(z.shape[0]).to(device)
    
    ts = torch.linspace(0, 1, n_steps).to(device)
    
    for t in ts:
        t_val = t.view(1, 1)
        x_t = path.sample_conditional_path(z, t_val, start_points=w)
        
        x_lig = x_t[:, :path.n_coords]
        h_lig = x_t[:, path.n_coords:]
        x_pocket = data.x_pocket.to(device)
        h_pocket = data.h_pocket.to(device)
        
        if unnorm_fn is not None:
             x_lig, x_pocket = unnorm_fn(x_lig, x_pocket)
        
        # Create a new Data object for this step
        data_t = Data(
            x_lig=x_lig.cpu(),
            h_lig=h_lig.cpu(),
            x_pocket=x_pocket.cpu(),
            h_pocket=h_pocket.cpu(),
            batch_lig=torch.zeros(x_lig.shape[0], dtype=torch.long),
            batch_pocket=torch.zeros(x_pocket.shape[0], dtype=torch.long),
        )
        trajectory.append(data_t)
        
    return trajectory

# Create and visualize the interpolation trajectory from the validation set
interp_traj = create_interpolation_trajectory(
    pocket_sample, 
    path, 
    n_steps=100, 
    unnorm_fn=model.unnorm_x if model.unnorm else None
)

visualize_trajectory(
    interp_traj, 
    image_path="outputs/trajectory_interpolation/mol_traj.png", 
    gif_path="outputs/mol_traj_interpolation.gif", 
    plot_every=2, 
    pymol_camera=camera,
    width=640,
    height=480
)

Visualizing trajectory: 100%|██████████| 100/100 [01:54<00:00,  1.14s/it]


In [10]:
pl.seed_everything(12) #5 9

res_traj = model.sample_with_trajectory(1, n_steps=100, force_connected=False, pocket=pocket_sample, n_atoms=15)
visualize_trajectory(res_traj, image_path="outputs/trajectory_gat_ast/mol_traj.png", gif_path="outputs/mol_traj_gat_ast.gif", plot_every=33, pymol_camera=camera, width=640, height=480)

Seed set to 12
Visualizing trajectory: 100%|██████████| 100/100 [00:08<00:00, 11.60it/s]


In [ ]:
res = model.sample(1, n_steps=20)

Seed set to 42


In [9]:
visualize_data(res, image_path="outputs/generated_molecules_pymol.png", session_path="outputs/generated_molecules_session.pse", width=2400, height=2400)

In [8]:
visualize_data_rdkit(res, image_path="outputs/generated_molecules.png", mols_per_row=2, sub_img_size=(1200, 1200))

Image saved to: outputs/generated_molecules.png


[16:46:19] Both bonds on one end of an atropisomer are on the same side - atoms are: 17 25


In [47]:
distances, other_metrics = model.test_distribution(
                test_distribution=model.val_distribution,
                metrics=[
                    "qed",
                    "logp",
                    "sas",
                ],  # TODO: add lipinski, but also in the val_dist
                atom_decoder=model.atom_decoder,
            )

Sampling molecules: 100%|██████████████████████████████████████████| 2/2 [00:20<00:00, 10.33s/batch]


In [49]:
model.normalization_params

{'mean': array([ 7.7809190e-08, -9.0558565e-08,  5.2221260e-08], dtype=float32),
 'std': array([5.8403964, 5.7269087, 5.8340626], dtype=float32)}

In [48]:
other_metrics

{'validity': tensor(1.), 'connectivity': tensor(0.), 'uniqueness': tensor(0.)}

In [ ]:
from torch_geometric.data import Data
from topoflow.utils.metrics import compute_connectivity
from topoflow.utils.molecules import data_to_rdmol

final_mol = res_traj[-1]
final_rdmol = data_to_rdmol(final_mol, atom_decoder=model.atom_decoder, sanitize=False)

connected = compute_connectivity(final_rdmol)[0]
print(connected)

[]


In [6]:
res_traj = model._test_integration(n_steps=80)
visualize_trajectory(res_traj, image_path="outputs/trajectory_test/mol_traj.png", gif_path="outputs/trajectory_test/mol_traj.gif", plot_every=2, width=2400, height=2400)

Visualizing trajectory:  86%|████████▋ | 69/80 [01:16<00:13,  1.26s/it][14:58:32] Both bonds on one end of an atropisomer are on the same side - atoms are: 1 21
[14:58:32] Both bonds on one end of an atropisomer are on the same side - atoms are: 27 28
Visualizing trajectory:  91%|█████████▏| 73/80 [01:21<00:08,  1.16s/it][14:58:36] The 2 defining bonds for an atropisomer are co-planar - atoms are: 26 23
[14:58:36] Both bonds on one end of an atropisomer are on the same side - atoms are: 22 23
Visualizing trajectory: 100%|██████████| 80/80 [01:26<00:00,  1.08s/it]
